In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

In [2]:
df = pd.DataFrame({
    'age': [25, None, 40, 35, None],
    'income': [30000, 45000, None, 80000, 52000],
    'city': ['Chennai', 'Mumbai', None, 'Chennai', 'Delhi'],
    'size': ['S', 'L', 'M', 'S', 'L']
})

print("Original DataFrame:")
print(df)
print("\\nMissing values in each column:")
print(df.isnull().sum())

Original DataFrame:
    age   income     city size
0  25.0  30000.0  Chennai    S
1   NaN  45000.0   Mumbai    L
2  40.0      NaN      NaN    M
3  35.0  80000.0  Chennai    S
4   NaN  52000.0    Delhi    L
\nMissing values in each column:
age       2
income    1
city      1
size      0
dtype: int64


In [3]:
df_simple = df.copy()

age_imputer = SimpleImputer(strategy='median')
df_simple[['age']] = age_imputer.fit_transform(df_simple[['age']])

income_imputer = SimpleImputer(strategy='median')
df_simple[['income']] = income_imputer.fit_transform(df_simple[['income']])

city_imputer = SimpleImputer(strategy='most_frequent')
df_simple[['city']] = city_imputer.fit_transform(df_simple[['city']])

print("DataFrame after SimpleImputer:")
print(df_simple)
print("\\nMissing values after imputation:")
print(df_simple.isnull().sum())

DataFrame after SimpleImputer:
    age   income     city size
0  25.0  30000.0  Chennai    S
1  35.0  45000.0   Mumbai    L
2  40.0  48500.0  Chennai    M
3  35.0  80000.0  Chennai    S
4  35.0  52000.0    Delhi    L
\nMissing values after imputation:
age       0
income    0
city      0
size      0
dtype: int64


In [4]:
numeric_columns = ['age', 'income']

knn_imputer = KNNImputer(n_neighbors=2)
df_knn = df.copy()
df_knn[numeric_columns] = knn_imputer.fit_transform(df_knn[numeric_columns])

print("Numeric columns after KNNImputer:")
print(df_knn[numeric_columns])

print("\\nNumeric columns after SimpleImputer (median):")
print(df_simple[numeric_columns])

print("\\nComparison of imputation methods:")
comparison = pd.DataFrame({
    'SimpleImputer_median_age': df_simple['age'],
    'KNNImputer_age': df_knn['age'],
    'SimpleImputer_median_income': df_simple['income'],
    'KNNImputer_income': df_knn['income']
})
print(comparison)

Numeric columns after KNNImputer:
    age   income
0  25.0  30000.0
1  30.0  45000.0
2  40.0  55000.0
3  35.0  80000.0
4  30.0  52000.0
\nNumeric columns after SimpleImputer (median):
    age   income
0  25.0  30000.0
1  35.0  45000.0
2  40.0  48500.0
3  35.0  80000.0
4  35.0  52000.0
\nComparison of imputation methods:
   SimpleImputer_median_age  KNNImputer_age  SimpleImputer_median_income  \
0                      25.0            25.0                      30000.0   
1                      35.0            30.0                      45000.0   
2                      40.0            40.0                      48500.0   
3                      35.0            35.0                      80000.0   
4                      35.0            30.0                      52000.0   

   KNNImputer_income  
0            30000.0  
1            45000.0  
2            55000.0  
3            80000.0  
4            52000.0  


In [5]:
df_encoded = df_simple.copy()

print("DataFrame before encoding:")
print(df_encoded)

DataFrame before encoding:
    age   income     city size
0  25.0  30000.0  Chennai    S
1  35.0  45000.0   Mumbai    L
2  40.0  48500.0  Chennai    M
3  35.0  80000.0  Chennai    S
4  35.0  52000.0    Delhi    L


In [6]:
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

city_encoded_array = onehot_encoder.fit_transform(df_encoded[['city']])
city_encoded_columns = onehot_encoder.get_feature_names_out(['city'])

city_encoded_df = pd.DataFrame(
    city_encoded_array,
    columns=city_encoded_columns,
    index=df_encoded.index
)

df_encoded = pd.concat(
    [df_encoded.drop(columns=['city']), city_encoded_df],
    axis=1
)

print("DataFrame after OneHotEncoding city:")
print(df_encoded)

DataFrame after OneHotEncoding city:
    age   income size  city_Chennai  city_Delhi  city_Mumbai
0  25.0  30000.0    S           1.0         0.0          0.0
1  35.0  45000.0    L           0.0         0.0          1.0
2  40.0  48500.0    M           1.0         0.0          0.0
3  35.0  80000.0    S           1.0         0.0          0.0
4  35.0  52000.0    L           0.0         1.0          0.0


In [7]:
ordinal_encoder = OrdinalEncoder(categories=[['S', 'M', 'L']])

df_encoded[['size']] = ordinal_encoder.fit_transform(df_encoded[['size']])

print("DataFrame after OrdinalEncoding size (S=0, M=1, L=2):")
print(df_encoded)

DataFrame after OrdinalEncoding size (S=0, M=1, L=2):
    age   income  size  city_Chennai  city_Delhi  city_Mumbai
0  25.0  30000.0   0.0           1.0         0.0          0.0
1  35.0  45000.0   2.0           0.0         0.0          1.0
2  40.0  48500.0   1.0           1.0         0.0          0.0
3  35.0  80000.0   0.0           1.0         0.0          0.0
4  35.0  52000.0   2.0           0.0         1.0          0.0


In [8]:
target = pd.Series(['Regular', 'Premium', 'Regular', 'Premium', 'Regular'],
                   name='customer_type')

label_encoder = LabelEncoder()
target_encoded = label_encoder.fit_transform(target)

target_df = pd.DataFrame({
    'customer_type': target,
    'customer_type_encoded': target_encoded
})

print("Original target and LabelEncoded target:")
print(target_df)
print("\\nClass mapping:")
for class_name, encoded_value in zip(label_encoder.classes_,
                                     label_encoder.transform(label_encoder.classes_)):
    print(f"{class_name} = {encoded_value}")

Original target and LabelEncoded target:
  customer_type  customer_type_encoded
0       Regular                      1
1       Premium                      0
2       Regular                      1
3       Premium                      0
4       Regular                      1
\nClass mapping:
Premium = 0
Regular = 1


In [9]:
print("Final encoded feature DataFrame:")
print(df_encoded)

print("\\nFinal DataFrame information:")
print(df_encoded.info())

Final encoded feature DataFrame:
    age   income  size  city_Chennai  city_Delhi  city_Mumbai
0  25.0  30000.0   0.0           1.0         0.0          0.0
1  35.0  45000.0   2.0           0.0         0.0          1.0
2  40.0  48500.0   1.0           1.0         0.0          0.0
3  35.0  80000.0   0.0           1.0         0.0          0.0
4  35.0  52000.0   2.0           0.0         1.0          0.0
\nFinal DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   age           5 non-null      float64
 1   income        5 non-null      float64
 2   size          5 non-null      float64
 3   city_Chennai  5 non-null      float64
 4   city_Delhi    5 non-null      float64
 5   city_Mumbai   5 non-null      float64
dtypes: float64(6)
memory usage: 372.0 bytes
None
